# A worked example: numfor

This notebook runs the same eight stages over a module from the corpus — `basic`, from
[numfor](https://github.com/numericfor/numfor) — and stays with it long
enough to say what the passing run does and does not establish.

Every code cell below is the shipped `translate` recipe, with no domain
extension installed and no stage configured — the single config file
involved sets the output directory and nothing else. Run it from a clone of this repository with the project environment built
(`uv sync`) and a Fortran compiler on PATH — the oracle stage compiles the
untouched Fortran with f2py.

The notebook itself asks nothing of its kernel beyond the standard library — any Jupyter front-end will do (e.g. `uvx --from jupyterlab jupyter lab`, no project dependency) — and every `%%bash` cell is a plain shell command that pastes into a terminal just as well.

In [1]:
# Run everything from the repository root, with the project venv on PATH.
import os
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "docs":
    ROOT = ROOT.parent
os.chdir(ROOT)
venv = ROOT / ".venv" / "bin"
if venv.is_dir() and str(venv) not in os.environ["PATH"]:
    os.environ["PATH"] = f"{venv}:{os.environ['PATH']}"
print(f"running from {ROOT.name}/")

running from RecastEngine/


## Getting the source in place

What the tool wants is the upstream repository at `corpus/numfor`, at the
commit the corpus pins. In a clone of this repository, where `corpus/numfor`
is registered as a submodule, that is one command (the cell below). Outside
one, clone it instead:

```bash
git clone https://github.com/numericfor/numfor.git corpus/numfor
git -C corpus/numfor checkout 65ee8b75b22ad300b54c87d632b3ebcd87de4b7c
```

Then stage the case:

In [2]:
%%bash
git submodule update --init --depth 1 corpus/numfor
python tools/corpus.py stage numfor

numfor: staged at output/numfor/staged
  recast run translate output/numfor/staged --config output/n

umfor/staged/recast.json


`stage` copies what `cases.json` lists — for `numfor`, the 133 `.f90` and
`.inc` files under `src/`, its test tree left out — flat into a fresh
`output/numfor/staged/`. Flat because an `#include "qtrs1d.inc"` names no
directory. `corpus/numfor` is only ever read from.

`stage` also leaves a `recast.json` beside the sources, pinning the run's
output to `output/numfor/`. The command below passes it.

## A successful run

In [3]:
%%bash
recast run translate output/numfor/staged \
    --config output/numfor/staged/recast.json --unit fortran:basic

fortran:basic
  [ok ] frontend   fortran                   
  [ok ] transform  translate.numpy      

       3 deferred block(s)
  [ok ] verifier   static.rwset                sampled: 57 blocks match
 

 [ok ] oracle     f2py-golden                 f2py:basic:6db7377c264c1f3e
  [ok ] verifier   differe

ntial.bitexact       bit_exact: 10 points across 1 subprogram(s), all bit-exact
  [ok ] verifier   s

ymbolic.notary             symbolic: no rewrites to notarize; the translation is print-order faithfu

...

63277a1ce2e1ace85afa6b977478b147f4580.json
  evidence: file:///Users/yueqichen/agent/SciRecast/Recas

tEngine/output/numfor/evidence/fortran_basic/e70c77077ab9c01a2f319175ce3c1ab68ab63d2fee18bbbcf4480d5

c5fb29ff9.json
  evidence: file:///Users/yueqichen/agent/SciRecast/RecastEngine/output/numfor/eviden

ce/fortran_basic/1791de4c57881efef0042983febe69638565aa2c59bd79673ca42a8505ab838f.json

1 unit(s), 3

 verdict(s), all passed


`basic` is numfor's 354-line utility module — kinds, a timer derived type,
a date stamp, `is_inf`. Two things to open under `output/numfor/`:

| | |
|---|---|
| `translate/fortran_basic/candidate/basic_numpy.py` | the generated Python, every block carrying the source lines it came from |
| `evidence/fortran_basic/*.json` | one manifest per verdict — artifact digest, oracle key, metrics |

### Three blocks the rules would not guess

The `3 deferred block(s)` are the rules declining to guess: two `cpu_time`
calls and one `date_and_time`. A refusal is
left standing in the output as a raise, tagged for whoever answers it:

In [4]:
%%bash
grep -n -A1 "AGENT_QUEUE" output/numfor/translate/fortran_basic/candidate/basic_numpy.py

797:    # B002 <- L200-L200 AGENT_QUEUE: intrinsic subroutine 'date_and_time' has no rule
798-    ra

ise NotImplementedError("intrinsic subroutine 'date_and_time' has no rule")  # B002
--
817:    # B00

6 <- L217-L217 AGENT_QUEUE: intrinsic subroutine 'cpu_time' has no rule
818-    raise NotImplemented

Error("intrinsic subroutine 'cpu_time' has no rule")  # B006
--
823:    # B001 <- L225-L225 AGENT_QU

EUE: intrinsic subroutine 'cpu_time' has no rule
824-    raise NotImplementedError("intrinsic subrou

tine 'cpu_time' has no rule")  # B001


That is not a wrong translation, and it is not a silent one. The other 57
blocks in the module are translated, and checked.

### How far the passing run actually reaches

`all passed` is a claim about three verifiers, and they do not cover the same
ground:

| verifier | what it covered |
|---|---|
| `static.rwset` | 57 of the module's 60 blocks — reads and writes agree with the source's |
| `differential.bitexact` | **one** subprogram, `is_inf`, 10 points, `max_ulp: 0` |
| `symbolic.notary` | 0 rewrites to notarize — the translation reorders no output |

The differential gate is the one that says a translation is *right*, and here
it saw one of the module's thirteen procedures. The reason is visibility, not
sampling: the reference is an f2py build of the untouched Fortran, f2py wraps
what the module makes public, and `basic.f90:74` declares `private` and then
exports exactly two procedures — `is_inf` and `print_msg`. `print_msg` holds
one of the three deferred blocks, so the gate skips it and says so
(`"skipped": ["print_msg"]`). The other eleven — the timer type's bound
procedures and the helpers around them — are private, and never reach the
oracle at all.

So: the module imports, its dataflow agrees with the source's, and the one
piece of it that could be executed against compiled Fortran matches bit for
bit. It does not say the timer procedures are correct. Nothing ran them.

### The evidence

Each verdict lands as its own content-addressed manifest — the `file://`
lines in the run output above. The differential one, trimmed:

In [5]:
import json
from pathlib import Path

manifests = [
    json.loads(p.read_text())
    for p in Path("output/numfor/evidence/fortran_basic").glob("*.json")
]
differential = max(
    (m for m in manifests if m["result"]["verifier"] == "differential.bitexact"),
    key=lambda m: m["timestamp"],
)
print(json.dumps(
    {
        "artifact": {
            k: differential["artifact"][k] for k in ("digest", "name", "transform")
        },
        "reference": differential["reference"],
        "result": {
            k: differential["result"][k]
            for k in ("verdict", "verifier", "passed", "metrics")
        },
    },
    indent=2,
))

{
  "artifact": {
    "digest": "c8f58b0a623cb44c1e664e61433579e2cf788af76949e82ab0a7ad688f5132df",
    "name": "fortran:basic",
    "transform": "recast.translate.fortran-to-numpy"
  },
  "reference": {
    "key": "f2py:basic:6db7377c264c1f3e",
    "oracle": "f2py-golden"
  },
  "result": {
    "verdict": "bit_exact",
    "verifier": "differential.bitexact",
    "passed": true,
    "metrics": {
      "bit_exact": 10,
      "max_rel": 0.0,
      "max_ulp": 0,
      "nan_mismatch": 0,
      "points": 10,
      "skipped": [
        "print_msg"
      ],
      "subprograms": {
        "is_inf": {
          "bit_exact": 10,
          "max_rel": 0.0,
          "max_ulp": 0,
          "nan_mismatch": 0,
          "points": 10
        }
      },
      "trials": 10
    }
  }
}


The digest is over the generated files, so the manifest names the artifact it
judged rather than the path it sat at. The oracle key folds the compiler's
version — which is why a manifest is a record of one run and is never
diffed against another's ([`examples/README.md`](../examples/) has the
long form of that argument).

## A failed run

`array_utils`, from the same case, translates and then does not get past
the first check:

In [6]:
%%bash
recast run translate output/numfor/staged \
    --config output/numfor/staged/recast.json --unit fortran:array_utils \
  || echo "recast exited $?" 

fortran:array_utils
  [ok ] frontend   fortran                   
  [ok ] transform  translate.numpy

             1 deferred block(s)
  [FAIL] verifier   static.rwset                failed: 3/69 blocks

 disagree: save_array1d/B005, save_array1d/B021, save_array2d/B016
  [ok ] store      fs-evidence   

              1 verdict(s) recorded
  evidence: file:///Users/yueqichen/agent/SciRecast/RecastEngine

/output/numfor/evidence/fortran_array_utils/7e48f70ae27db6111d9cf9271bd52b3ad0979e65089ef1e1288382a9

6b5e7f06.json

1 unit(s), 1 verdict(s), FAILED


recast exited 1


The run exits 1, and the stages after the failed verifier never run: no f2py
build, no differential, one verdict recorded instead of three. The manifest
names what disagreed, per block and per symbol:

In [7]:
import json
from pathlib import Path

manifests = [
    json.loads(p.read_text())
    for p in Path("output/numfor/evidence/fortran_array_utils").glob("*.json")
]
rwset = max(
    (m for m in manifests if m["result"]["verifier"] == "static.rwset"),
    key=lambda m: m["timestamp"],
)
for failure in rwset["result"]["metrics"]["failures"]:
    print(json.dumps(failure))

{"block": "save_array1d/B005", "reads_source_only": ["reshape"], "reads_target_only": [], "writes_source_only": [], "writes_target_only": []}
{"block": "save_array1d/B021", "reads_source_only": ["unit_"], "reads_target_only": [], "writes_source_only": [], "writes_target_only": []}
{"block": "save_array2d/B016", "reads_source_only": ["unit_"], "reads_target_only": [], "writes_source_only": [], "writes_target_only": []}


The last two source blocks read `unit_`; the translated ones do not (the
first counts `reshape` the same one-sided way). The gate does
not say which side is wrong — a defect in the translation, or a limit of the
read/write analysis — and does not need to. It fails closed, and 66 of the
69 blocks matching does not buy the other three.